In [12]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import time

kat = players.find_players_by_full_name("Karl-Anthony Towns")[0]
kat_id = kat["id"]

seasons = ["2022-23", "2023-24", "2024-25", "2025-26"]

all_logs = []

for season in seasons:
    for season_type in ["Regular Season", "Playoffs"]:
        try:
            log = playergamelog.PlayerGameLog(
                player_id=kat_id,
                season=season,
                season_type_all_star=season_type
            ).get_data_frames()[0]

            log["SEASON"] = season
            log["SEASON_TYPE"] = season_type

            all_logs.append(log)

            time.sleep(0.6)

        except Exception as e:
            print(f"Failed {season} {season_type}: {e}")

kat_all = pd.concat(all_logs, ignore_index=True)

print(kat_all.shape)
print(kat_all[["SEASON", "SEASON_TYPE", "GAME_DATE", "MATCHUP", "BLK"]].head())


(291, 29)
    SEASON     SEASON_TYPE     GAME_DATE      MATCHUP  BLK
0  2022-23  Regular Season  Apr 09, 2023  MIN vs. NOP    1
1  2022-23  Regular Season  Apr 08, 2023    MIN @ SAS    2
2  2022-23  Regular Season  Apr 04, 2023    MIN @ BKN    0
3  2022-23  Regular Season  Apr 02, 2023  MIN vs. POR    0
4  2022-23  Regular Season  Mar 31, 2023  MIN vs. LAL    0


In [14]:
Kat_gamelog_df = kat_all[[
    "SEASON",
    "GAME_DATE",
    "MATCHUP",
    "WL",
    "MIN",
    "BLK",
    "SEASON_TYPE"
]].copy()

Kat_gamelog_df["GAME_DATE"] = pd.to_datetime(Kat_gamelog_df["GAME_DATE"])

# Opponent abbreviation
Kat_gamelog_df["OPP"] = Kat_gamelog_df["MATCHUP"].str.strip().str[-3:]

# Home = 1, away = 0
Kat_gamelog_df["HOME"] = Kat_gamelog_df["MATCHUP"].str.contains("vs.").astype(int)

# Binary target for block prop
Kat_gamelog_df["Block_Stat"] = (Kat_gamelog_df["BLK"] > 0).astype(int)

# Sort oldest to newest so EMA works correctly later
Kat_gamelog_df = Kat_gamelog_df.sort_values("GAME_DATE").reset_index(drop=True)

Kat_gamelog_df = Kat_gamelog_df.drop(columns= ['MATCHUP', 'WL'])

print(Kat_gamelog_df.tail(15))
print(Kat_gamelog_df["SEASON_TYPE"].value_counts())
print(Kat_gamelog_df["OPP"].value_counts())

      SEASON  GAME_DATE  MIN  BLK     SEASON_TYPE  OPP  HOME  Block_Stat
276  2025-26 2026-04-10   30    0  Regular Season  TOR     1           0
277  2025-26 2026-04-18   33    3        Playoffs  ATL     1           1
278  2025-26 2026-04-20   34    2        Playoffs  ATL     1           1
279  2025-26 2026-04-23   34    2        Playoffs  ATL     0           1
280  2025-26 2026-04-25   29    0        Playoffs  ATL     0           0
281  2025-26 2026-04-28   34    2        Playoffs  ATL     1           1
282  2025-26 2026-04-30   28    1        Playoffs  ATL     0           1
283  2025-26 2026-05-04   20    2        Playoffs  PHI     1           1
284  2025-26 2026-05-06   27    0        Playoffs  PHI     1           0
285  2025-26 2026-05-08   26    1        Playoffs  PHI     0           1
286  2025-26 2026-05-10   20    2        Playoffs  PHI     0           1
287  2025-26 2026-05-19   40    1        Playoffs  CLE     1           1
288  2025-26 2026-05-21   36    0        Playoffs  

In [15]:
from nba_api.stats.endpoints import leaguedashteamshotlocations
import pandas as pd
import time

def get_team_rim_fga(season, season_type="Regular Season"):
    shot_df = leaguedashteamshotlocations.LeagueDashTeamShotLocations(
        season=season,
        season_type_all_star=season_type,
        per_mode_detailed="PerGame",
        distance_range="By Zone"
    ).get_data_frames()[0]

    print("Columns from NBA API:")
    print(shot_df.columns.tolist())

    # Find team abbreviation column
    team_col = None
    for col in shot_df.columns:
        if "TEAM_ABBREVIATION" in str(col):
            team_col = col
            break

    if team_col is None:
        raise ValueError("Could not find TEAM_ABBREVIATION column")

    # Find Restricted Area FGA column
    rim_fga_col = None
    for col in shot_df.columns:
        col_str = str(col).lower()
        if "restricted area" in col_str and "fga" in col_str:
            rim_fga_col = col
            break

    if rim_fga_col is None:
        raise ValueError("Could not find Restricted Area FGA column")

    rim_df = shot_df[[team_col, rim_fga_col]].copy()
    rim_df.columns = ["OPP", "OPP_RIM_FGA"]

    return rim_df

In [17]:
from nba_api.stats.static import teams

nba_teams = teams.get_teams()

abbr_to_name = {
    team["abbreviation"]: team["full_name"]
    for team in nba_teams
}

name_to_abbr = {
    team["full_name"]: team["abbreviation"]
    for team in nba_teams
}

print(abbr_to_name)

{'ATL': 'Atlanta Hawks', 'BOS': 'Boston Celtics', 'CLE': 'Cleveland Cavaliers', 'NOP': 'New Orleans Pelicans', 'CHI': 'Chicago Bulls', 'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets', 'GSW': 'Golden State Warriors', 'HOU': 'Houston Rockets', 'LAC': 'Los Angeles Clippers', 'LAL': 'Los Angeles Lakers', 'MIA': 'Miami Heat', 'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves', 'BKN': 'Brooklyn Nets', 'NYK': 'New York Knicks', 'ORL': 'Orlando Magic', 'IND': 'Indiana Pacers', 'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns', 'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings', 'SAS': 'San Antonio Spurs', 'OKC': 'Oklahoma City Thunder', 'TOR': 'Toronto Raptors', 'UTA': 'Utah Jazz', 'MEM': 'Memphis Grizzlies', 'WAS': 'Washington Wizards', 'DET': 'Detroit Pistons', 'CHA': 'Charlotte Hornets'}
